# Audio Signal Analysis & Sound Classification

We will work with **raw signals** — audio waveforms. An audio file is just a **1D numpy array** of air pressure values sampled thousands of times per second. Everything you've learned (numpy, matplotlib, pandas, scikit-learn) still applies.

By the end of this session you will:
- Load and visualize audio waveforms
- Decompose sounds into frequencies using the **FFT** 
- Create and read **spectrograms**
- Extract audio features
- Build an ML model that **classifies environmental sounds**

In [ ]:
!pip install librosa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
import pandas as pd

In [ ]:
import os, zipfile, urllib.request


def download_and_extract_esc50():
    esc50_dir = './ESC-50-master'

    if not os.path.exists(esc50_dir):
        print("Downloading ESC-50 dataset (this takes ~1 minute)...")
        url = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
        zip_path = "ESC-50.zip"
        urllib.request.urlretrieve(url, zip_path)
        
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('.')
        os.remove(zip_path)
        print("Done!")
    else:
        print("ESC-50 already downloaded.")

    # Load the metadata
    meta = pd.read_csv(os.path.join(esc50_dir, 'meta', 'esc50.csv'))
    print(f"\nTotal clips: {len(meta)}")
    print(f"Categories: {meta['category'].nunique()}")
    print(f"\nAll categories:")
    for i, cat in enumerate(sorted(meta['category'].unique()), 1):
        print(f"  {i:2d}. {cat}")

    return esc50_dir, meta

esc50_dir, meta = download_and_extract_esc50()

## Part 1: Audio is Just a Numpy Array

In [ ]:
# Generate audio signal from scratch
# Generate a 440 Hz tone (A4 — the tuning note)
sr = 22050          # Sample rate (samples per second)
duration = 2.0      # seconds
f = 440             # frequency in Hz

t = np.linspace(0, duration, int(sr * duration), endpoint=False)
tone_440 = np.sin(2 * np.pi * f * t)

print(f"Sample rate: {sr} Hz")
print(f"Duration: {duration} s")
print(f"Array shape: {tone_440.shape}")
print(f"That's {len(tone_440)} numbers — one per sample")

# Listen to it!
ipd.display(ipd.Audio(tone_440, rate=sr))

In [ ]:
# Plot the waveform
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# Full waveform
axes[0].plot(t, tone_440)
axes[0].set_title('Full Waveform (2 seconds)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Zoom in to see individual oscillations
zoom = int(sr * 0.01)  # First 10 ms
axes[1].plot(t[:zoom], tone_440[:zoom])
axes[1].set_title('Zoomed In (first 10 ms)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

# How many complete cycles in 10 ms at 440 Hz?
print(f"Expected cycles in 10ms: {440 * 0.01} = {440 * 0.01:.1f}")

### Exercise 1: Build a chord

A musical chord is just multiple sine waves added together.

Create a C major chord by adding three frequencies:
- C4 = 261.6 Hz
- E4 = 329.6 Hz  
- G4 = 392.0 Hz

Then listen to it. How does it sound compared to the single tone?

In [ ]:
# YOUR CODE HERE: Create a C major chord


### Loading real audio files

Now let's work with actual recorded sounds. We'll use `librosa.load()` which returns two things:
1. `y` — the audio signal as a numpy array
2. `sr` — the sample rate

Loading someone's voice: "Hello my friend"

In [ ]:
path = r'hello_my_friend.mp3'

y, sr = librosa.load(path, sr=None)

print(f"Audio array shape: {y.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {len(y) / sr:.2f} seconds")

# Listen
ipd.display(ipd.Audio(y, rate=sr))

# Plot
plt.figure(figsize=(8, 2))
librosa.display.waveshow(y, sr=sr)
plt.title('Kou Waveform')
plt.tight_layout()
plt.show()

> Try make this into a function!

## Part 2: Frequency Decomposition — FFT and Spectrograms

You know from your DSP course that the **Discrete Fourier Transform** decomposes a signal into its frequency components:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-j2\pi kn/N}$$

In Python, this is one line: `np.fft.fft()`. Let's see it in action.

### Demo: Decomposing a known signal

First, let's verify FFT works on signals we constructed ourselves:

In [ ]:
# Create a signal with known frequencies: 200 Hz + 500 Hz + 1200 Hz
sr_ = 22050
duration = 1.0
t = np.linspace(0, duration, int(sr_ * duration), endpoint=False)

f1, f2, f3 = 200, 500, 1200
signal = np.sin(2 * np.pi * f1 * t) + 0.7 * np.sin(2 * np.pi * f2 * t) + 0.3 * np.sin(2 * np.pi * f3 * t)

# Listen to the composite signal
ipd.display(ipd.Audio(signal, rate=sr_))

# Compute FFT
fft_result = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal), d=1/sr_)

# We only need the positive half (signal is real-valued)
positive_mask = frequencies >= 0
freqs_pos = frequencies[positive_mask]
magnitude = np.abs(fft_result[positive_mask])

# Plot
fig, axes = plt.subplots(2, 1, figsize=(8,4))

# Time domain
axes[0].plot(t[:2000], signal[:2000])
axes[0].set_title('Time Domain: Composite Signal (first ~90 ms)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Frequency domain
axes[1].plot(freqs_pos, magnitude)
axes[1].set_xlim(0, 2000)
axes[1].set_title('Frequency Domain (FFT)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')

# Mark the expected peaks
for f in [f1, f2, f3]:
    axes[1].axvline(x=f, color='red', linestyle='--', alpha=0.5, label=f'{f} Hz')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"The FFT correctly recovers our three frequencies: {f1}, {f2}, {f3} Hz!")

### Excercise 2: Show the FT of yours and your mates voice

In [ ]:
## Your code HERE!

### FFT -> STFT to capture the temporal signature

In [ ]:
# Spectrogram of the trumpet
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spectrogram
S_kou = librosa.stft(y)
S_db_kou = librosa.amplitude_to_db(np.abs(S_kou), ref=np.max)
img1 = librosa.display.specshow(S_db_kou, sr=sr, x_axis='time', y_axis='hz', ax=axes[0])
axes[0].set_title('Kou Spectrogram')
axes[0].set_ylim(0, 5000)
fig.colorbar(img1, ax=axes[0])

### Mel spectrogram

In [ ]:
# Mel spectrogram — the standard representation for audio ML
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

S_mel_kou = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
S_mel_db_kou = librosa.power_to_db(S_mel_kou, ref=np.max)
img1 = librosa.display.specshow(S_mel_db_kou, sr=sr, x_axis='time', y_axis='mel', ax=axes[0])
axes[0].set_title('Kou — Mel Spectrogram')
fig.colorbar(img1, ax=axes[0])

## Part 3: Reverse Engineer the sound

### Can we convert the spectrogram to sound?

In [ ]:
# Undo dB conversion
S_mel_recovered = librosa.db_to_power(S_mel_db_kou)

# Invert mel filterbank back to linear spectrogram
S_linear = librosa.feature.inverse.mel_to_stft(S_mel_recovered, sr=sr, n_fft=2048)


y_recovered = librosa.istft(S_linear)

ipd.display(ipd.Audio(y_recovered, rate=sr))

> make a function to convert the mel_db to sound ready to play

In [ ]:
# Phase retrieval using Griffin-Lim algorithm
y_griffin = librosa.griffinlim(S_linear, n_iter=64)
ipd.display(ipd.Audio(y_griffin, rate=sr))

### Can we convert a hand-drawn image into sound?

In [ ]:
# import napari
import napari

viewer = napari.Viewer()

# Create a 3D array to represent the spectrogram as an image
S_mel_db_blank = np.ones_like(S_mel_db_kou, dtype=np.uint8)
viewer.add_labels(S_mel_db_blank)

In [ ]:
# Read the label
S_mel_db_modified = viewer.layers[0].data

In [ ]:
plt.imshow(S_mel_db_modified)

### Sound decomposition!

## Part 4: Extracting audio features


In [ ]:
# Extract features from the trumpet clip
def extract_features(y, sr):
    """Extract audio features from a signal."""
    features = {}
    
    # MFCCs — 13 coefficients, averaged over time
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc_{i+1}'] = np.mean(mfccs[i])
    
    # Spectral centroid
    features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    
    # Zero-crossing rate
    features['zero_crossing_rate'] = np.mean(librosa.feature.zero_crossing_rate(y))
    
    # Spectral bandwidth
    features['spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    
    # RMS energy
    features['rms_energy'] = np.mean(librosa.feature.rms(y=y))
    
    # Spectral rolloff
    features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    return features



### Visualizing MFCCs over time

MFCCs are the most important features in audio ML. Let's see what they look like as a time series:

In [ ]:
# MFCCs over time for trumpet vs piano
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
img1 = librosa.display.specshow(mfccs, sr=sr, x_axis='time', ax=axes[0])
axes[0].set_title('MFCCs over Time')
axes[0].set_ylabel('MFCC Coefficient')
fig.colorbar(img1, ax=axes[0])

## Part 5: Building a Sound Event Classifier

### The ML pipeline

```
1. Load audio files
2. Extract features from each clip
3. Train/test split
4. Train classifier
5. Evaluate with confusion matrix
```

In [ ]:
esc50_dir, meta = download_and_extract_esc50()

In [ ]:
# Select 10 interesting categories for our classifier
selected_categories = [
    'dog', 'rain', 'clock_tick', 'siren', 'clapping',
    'helicopter', 'sea_waves', 'sneezing', 'engine', 'cat'
]

meta_subset = meta[meta['category'].isin(selected_categories)].copy()
print(f"Selected {len(meta_subset)} clips across {len(selected_categories)} categories")
print(meta_subset['category'].value_counts())

In [ ]:
# Play one example from each category
audio_dir = os.path.join(esc50_dir, 'audio')

fig, axes = plt.subplots(2, 5, figsize=(18, 5))
axes = axes.flatten()

for i, category in enumerate(selected_categories):
    # Get the first clip in this category
    row = meta_subset[meta_subset['category'] == category].iloc[0]
    filepath = os.path.join(audio_dir, row['filename'])
    
    y, sr = librosa.load(filepath, sr=22050)
    
    # Plot waveform
    librosa.display.waveshow(y, sr=sr, ax=axes[i])
    axes[i].set_title(category, fontsize=10)
    axes[i].set_xlabel('')
    
    # Display audio player (only for first 5 to save space)
    if i < 5:
        print(f"--- {category} ---")
        ipd.display(ipd.Audio(y, rate=sr))

plt.tight_layout()
plt.show()

In [ ]:
# Extract features from all selected clips
audio_dir = os.path.join(esc50_dir, 'audio')

all_features = []
all_labels = []
failed = 0

print(f"Extracting features from {len(meta_subset)} clips...")
for idx, (_, row) in enumerate(meta_subset.iterrows()):
    filepath = os.path.join(audio_dir, row['filename'])
    try:
        y, sr = librosa.load(filepath, sr=22050, duration=5.0)
        feats = extract_features(y, sr)
        all_features.append(feats)
        all_labels.append(row['category'])
    except Exception as e:
        failed += 1
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(meta_subset)} clips...")

# Create DataFrame
features_df = pd.DataFrame(all_features)
features_df['category'] = all_labels

print(f"\nDone! Extracted features from {len(features_df)} clips ({failed} failed)")
print(f"Feature matrix shape: {features_df.shape}")
features_df.head()

### Implementing RandomForestClassifier

In [ ]:
# Prepare for ML

In [ ]:
# Train a Random Forest classifier

In [ ]:
# Confusion matrix — shows exactly where the model gets confused

In [ ]:
# Which features matter most?


## Part 6: Challenge: Classify Your Own Sounds!

### Either: Test on the data in the dataset, or your own sound!